# editions

> verses laid into titles, a run's numbering reported, a reading checked against a witness edition, two readings of a page compared

In [ ]:
#| default_exp editions

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
import re
from difflib import SequenceMatcher
from fastcore.all import first
from lekhaka.verses import vnum, blocks, kind, virama, deva

These take the chapters `assemble` returns, `[{heading, colophon, verses}]`, or a page of blocks. A second edition is data the caller has fetched or loaded: nothing here reads a file or the web.

## Titles

`cut` lays chapters' verses into titles `{name, title, chapter, verses}`. `how='per'` makes a title of each `per` verses by the verses' own numbers, the unnumbered verses before the first numbered one going with it; `'chapters'` a title a chapter; `'one'` a single title. An unnumbered verse is lettered within its own run (`letters`, क ख ग …) and carries `word`, `open_word`, so it never takes a numbered verse's citation. `digits` writes the numbers in a title; Devanagari by default. `numbering` reports what a run of verses' own numbers say.

In [ ]:
#| export
LETTERS = 'कखगघङचछजझञ'
def _tag(vs, letters, word):
    "Verses with their numbers read, each unnumbered one lettered within its run and given `word`."
    out, k = [], 0
    for v in vs:
        if (n := vnum(v['n'])) is None: out.append(dict(v, n=letters[k % len(letters)], word=word)); k += 1
        else: k = 0; out.append(dict(v, n=n))
    return out

def cut(chs, how:str='per', per:int=10, open_word:str='मङ्गलम्', stem:str='work', book:str='', digits=deva, letters:str=LETTERS) -> list:
    "Chapters' verses laid into titles `[{name, title, chapter, verses}]`: `per` verses a title by their numbers, a title a chapter, or `one`."
    tag = lambda vs: _tag(vs, letters, open_word)
    if how == 'chapters':
        return [dict(name=f'{stem}_{i:02d}', title=' '.join(x for x in (book, digits(i), c.get('heading')) if x), chapter=i, verses=tag(c['verses'])) for i, c in enumerate(chs, 1)]
    vs = [v for c in chs for v in c['verses']]
    if how == 'one': return [dict(name=stem, title=book, chapter=1, verses=tag(vs))]
    groups, cur, hi = [], [], None
    for v in vs:
        n = vnum(v['n'])
        if n is not None and hi is not None and n > hi: groups.append(cur); cur = []
        if n is not None and not any(vnum(x['n']) is not None for x in cur): hi = ((n - 1) // per + 1) * per
        cur.append(v)
    if cur: groups.append(cur)
    out = []
    for g in groups:
        ns = [n for v in g if (n := vnum(v['n'])) is not None]
        a, b = ((ns[0] - 1) // per * per + 1, min((ns[0] - 1) // per * per + per, max(ns))) if ns else (0, 0)
        out.append(dict(name=f'{stem}_{a:03d}_{b:03d}', title=f'{book} {digits(a)}–{digits(b)}'.strip(), chapter=1, verses=tag(g)))
    return out

def numbering(vs) -> dict:
    "What a run of verses' own numbers say: how many are numbered and unnumbered, the numbers skipped and repeated, the first and last."
    ns = [n for v in vs if (n := vnum(v['n'])) is not None]
    seen, rep = set(), []
    for n in ns:
        if n in seen: rep.append(n)
        seen.add(n)
    gaps = [k for a, b in zip(ns, ns[1:]) if b > a + 1 for k in range(a + 1, b)]
    return dict(numbered=len(ns), unnumbered=len(vs) - len(ns), skipped=gaps, repeated=rep, first=first(ns), last=ns[-1] if ns else None)

In [ ]:
V = lambda n, t='x': dict(n=n, text=t, page=1, heading=None)
_cs = [dict(heading='प्रथमः', verses=[V(None), V(None), V('१'), V(2)]), dict(heading='द्वितीयः', verses=[V(3), V(None), V('௧௧'), V(12)])]
test_eq([(t['name'], t['title'], [v['n'] for v in t['verses']]) for t in cut(_cs, per=10, book='शतकम्')],
        [('work_001_003', 'शतकम् १–३', ['क', 'ख', 1, 2, 3, 'क']), ('work_011_012', 'शतकम् ११–१२', [11, 12])])
test_eq(cut(_cs)[0]['verses'][0]['word'], 'मङ्गलम्'); test_eq('word' in cut(_cs)[0]['verses'][2], False)
test_eq([(t['name'], t['title']) for t in cut(_cs, 'chapters', stem='s', book='ग्रन्थः')], [('s_01', 'ग्रन्थः १ प्रथमः'), ('s_02', 'ग्रन्थः २ द्वितीयः')])
test_eq(len(cut(_cs, 'one')[0]['verses']), 8)
test_eq(numbering([V(1), V(None), V('३'), V('௩'), V(6)]), dict(numbered=4, unnumbered=1, skipped=[2, 4, 5], repeated=[3], first=1, last=6))

## A witness

`witness_verses` cuts a witness edition's raw page (a sanskritdocuments page, a GRETIL file) into `[(n, text)]` at each `॥ n ॥` or `|| n ||`, markup dropped, a few lines a verse. It is to proofread against, never to copy from. `translit`, given, reads an IAST page as Devanagari. `letters` keeps only a text's Devanagari letters, the closing `॥ n ॥` and any bracketed note gone: what the checks compare.

`text_changes` makes the cheap checks of a born-digital text: each line's dropped pāda-final virāma (`virama`, by `rule`), and for each verse `metre` names no metre for, the nearest witness verse (letters alike at `at` or more) word by word, where it spells otherwise (by `witness`, and `metre` too when the verse then scans). Each change is `{id, chapter, n, page, read, print, why, by}`, a row a proofreader accepts or not, and a `fix_chapters` row once accepted.

In [ ]:
#| export
def witness_verses(raw:str, translit=None) -> list:
    "A witness edition's verses `[(n, text)]` from its raw page: markup gone, cut at each `॥ n ॥` or `|| n ||`; `translit` reads IAST as Devanagari."
    t = re.sub(r'<[^>]+>', '\n', re.sub(r'(?is)<(script|style)[^>]*>.*?</\1>', '', raw)).replace('&nbsp;', ' ')
    t = re.sub(r'\|\|\s*[A-Za-z]+_[\d.]*?(\d+)\s*\|\|', r'@@\1@@', t)   # GRETIL's `|| BharSt_1.12 ||`: the verse's own number
    if translit and len(re.findall(r'[ऀ-ॿ]', t)) < 200: t = translit(t.replace('-', '').replace('||', '॥').replace('|', '।'))   # GRETIL hyphenates compounds
    t = re.sub(r'@@(\d+)@@', r'॥ \1 ॥', t).replace('||', '॥').replace('।।', '॥')
    out, last = [], 0
    for m in re.finditer(r'॥\s*(?:[^\s॥]*?[._])?(\d+)\s*॥', t):
        out.append((vnum(m[1]), '\n'.join([l for l in t[last:m.start()].strip().split('\n') if l.strip()][-6:]))); last = m.end()   # a page's header stays out
    return out

def letters(s:str) -> str:
    "`s`'s Devanagari letters alone: its closing `॥ n ॥`, bracketed notes, spaces, daṇḍas and digits gone."
    return re.sub(r'[^ऀ-ॣ]', '', re.sub(r'\([^)]*\)', '', re.sub(r'॥[^॥]*॥', '', s or '')))

def _words(t): return re.findall(r'[ऀ-ॣ]+', t)

def text_changes(chs, wit=(), metre=None, at:float=.7, key=None) -> list:
    "Dropped pāda-final virāmas, and where the nearest witness verse spells otherwise for a verse `metre` names none for, as changes to accept."
    key, wv, out = key or (lambda i, c: i), [(n, t, letters(t)) for n, t in wit], []
    for ci, c in enumerate(chs, 1):
        ch = key(ci, c)
        for v in c['verses']:
            n, pg, ls = v['n'], v.get('page'), [l for l in v['text'].split('\n') if l.strip()]
            for li, l in enumerate(ls):
                l = re.sub(r'\s*॥[^॥]*॥?\s*$', '', l)   # the closing ॥ n ॥ off, so a verse's last word ends the line
                if (x := virama(l)) != l:
                    a = next(w for w, y in zip(_words(l) + [''], _words(x) + ['']) if w != y)
                    out.append(dict(id=f'{ch}.{n}.v{li}', chapter=ch, n=n, page=pg, read=a, print=a + '्', why='pāda-final virāma', by=['rule']))
            if not wv or not ls or (metre and metre(ls)): continue
            txt = ' '.join(ls)
            sc, wn, wt = max(((SequenceMatcher(None, letters(txt), x, autojunk=False).ratio(), wn, t) for wn, t, x in wv), default=(0, None, ''))
            if sc < at: continue
            a, b = _words(txt), _words(wt)
            for op, i1, i2, j1, j2 in SequenceMatcher(None, a, b, autojunk=False).get_opcodes():
                if op != 'replace' or i2 - i1 > 2: continue
                ra, pb = ' '.join(a[i1:i2]), ' '.join(b[j1:j2])
                if ra not in txt: continue
                scans = bool(metre and metre([l.replace(ra, pb, 1) for l in ls]))
                out.append(dict(id=f'{ch}.{n}.w{i1}', chapter=ch, n=n, page=pg, read=ra, print=pb, by=['witness'] + (['metre'] if scans else []),
                                why=('no metre named; ' if metre else '') + f'the witness ({wn}) reads so' + (', and the verse then scans' if scans else '')))
    return out

In [ ]:
_raw = '<html><script>x()</script><p>नीतिशतकम्</p><p>दिक्कालाद्यनवच्छिन्नानन्तचिन्मात्रमूर्तये ।<br>स्वानुभूत्येकमानाय नमः शान्ताय तेजसे ॥ १॥</p><p>बोद्धारो मत्सरग्रस्ताः प्रभवः स्मयदूषिताः ।<br>अबोधोपहताश्चान्ये जीर्णमङ्गे सुभाषितम् ॥ 2 ॥</p></html>'
_w = witness_verses(_raw)
test_eq([n for n, _ in _w], [1, 2]); test_eq(_w[1][1], 'बोद्धारो मत्सरग्रस्ताः प्रभवः स्मयदूषिताः ।\nअबोधोपहताश्चान्ये जीर्णमङ्गे सुभाषितम्')
test_eq([n for n, _ in witness_verses('a b c || BharSt_1.12 || d e f || BharSt_1.13 ||')], [12, 13])
test_eq([t for _, t in witness_verses('rama || 1 ||', translit=str.upper)], ['RAMA'])   # a short Latin page goes through `translit`
test_eq(letters('राम (रामः) ॥ १ ॥ सीता ।'), 'रामसीता')
_ch = [dict(verses=[dict(n=1, page='p1', text='दिक्कालाद्यनवच्छिन्नानन्तचिन्मात्रमूर्तये ।\nस्वानुभूत्येकमानाय नमः शान्ताय तेजसे ॥ १॥'),
                    dict(n=2, page='p1', text='बोद्धारो मत्सरग्रस्ताः प्रभवः स्मयदूषिताः ।\nअबोधोपहताश्चान्ये जीर्णमङ्गे सुभाषितम ॥ २॥')])]
_x = text_changes(_ch, _w)
test_eq([(x['id'], x['read'], x['print'], x['by']) for x in _x], [('1.2.w6', 'सुभाषितम', 'सुभाषितम्', ['witness'])])
test_eq([(x['id'], x['read'], x['print'], x['by']) for x in text_changes([dict(verses=[dict(n=1, text='नमो भगवत ।\nवासुदेवाय ॥ १॥')])])], [('1.1.v0', 'भगवत', 'भगवत्', ['rule'])])
test_eq([x['by'] for x in text_changes(_ch, _w, metre=lambda ls: 'anuṣṭubh' if 'सुभाषितम्' in ls[-1] else None)], [['witness', 'metre']])

## Against a reference edition

`ref_check` flags, without changing anything, a verse unlike every reference verse near it by place and by number (letters alike under `at`), and a verse with a lacuna (····). `ref` is `{chapter: [letters of each verse]}`, made with `letters`. `move_lines` mends what a reader's cut between verses gets wrong, against `ref` `{(chapter, n): text}` folded by `fold` (`letters`, or an IAST folding for an IAST reference): leading lines of verse n that close verse n-1 move back, and leading lines that match no verse near (an editor's quotation) are dropped; a verse whose first lines read unlike the reference otherwise is flagged to check by hand. `key(i, chapter)` names a chapter in both, its place from 1 by default.

In [ ]:
#| export
def ref_check(chs, ref, probs:list=None, at:float=.8, win:int=6, key=None) -> list:
    "Problems for a proofreader: verses unlike every reference verse near them by place and number, and lacunae; nothing changed."
    probs, key = [] if probs is None else probs, key or (lambda i, c: i)
    for ci, c in enumerate(chs, 1):
        ch = key(ci, c); rs = ref.get(ch, [])
        for i, v in enumerate(c['verses']):
            n, t, pg = v['n'], v['text'], v.get('page')
            if '··' in t or '…' in t: probs.append(f'{ch}.{n} ({pg}): the print leaves a lacuna (····)')
            near = rs[max(0, i - win):i + win] + rs[max(0, n - 1 - win):n - 1 + win]
            if not near or (i >= len(rs) and n > len(rs)): continue   # past where a short reference stops
            if (sc := max(SequenceMatcher(None, letters(t), r).ratio() for r in near)) < at: probs.append(f'{ch}.{n} ({pg}): reads unlike the reference ({sc:.2f}), check against the scan')
    return probs

def move_lines(chs, ref, probs:list=None, fold=letters, key=None) -> list:
    "Chapters with leading lines of verse n that close verse n-1 moved back, and leading lines matching no verse dropped, against `ref`."
    probs, key = [] if probs is None else probs, key or (lambda i, c: i)
    sim = lambda ls, r: SequenceMatcher(None, ''.join(fold(l) for l in ls), r).ratio() if ls and r else 0
    for ci, c in enumerate(chs, 1):
        ch, vs = key(ci, c), c['verses']
        for i, v in enumerate(vs):
            n = v['n']; r, ls = ref.get((ch, n)), v['text'].split('\n')
            if not r or len(ls) < 2: continue
            whole = sim(ls, r)
            gain, j = max(((sim(ls[j:], r) - whole, j) for j in range(1, len(ls))), default=(0, 0))
            if gain <= .12: continue
            head, rp = ls[:j], ref.get((ch, n - 1))
            if i and vs[i - 1]['n'] == n - 1 and rp and sim(vs[i - 1]['text'].split('\n') + head, rp) > sim(vs[i - 1]['text'].split('\n'), rp) + .03:
                vs[i - 1] = dict(vs[i - 1], text=vs[i - 1]['text'] + '\n' + '\n'.join(head)); probs.append(f'{ch}.{n}: {j} leading line(s) moved back to {ch}.{n - 1}')
            elif whole >= .6 and whole + gain >= .9 and max(sim(head, ref.get((ch, m))) for m in range(n - 2, n + 3)) < .4:
                probs.append(f'{ch}.{n}: {j} leading line(s) dropped, matching no verse (a quotation): {head[0][:40]}')
            else: probs.append(f'{ch}.{n}: check by hand; its first {j} line(s) read unlike the reference: {head[0][:40]}'); continue
            vs[i] = dict(v, text='\n'.join(ls[j:]))
    return chs

In [ ]:
import json
from lekhaka.verses import assemble

In [ ]:
_bj, _ = assemble(json.load(open('samples/brihat_jataka.json')))
_ref = {ci: [letters(v['text']) for v in c['verses']] for ci, c in enumerate(_bj, 1)}
test_eq(ref_check(_bj, _ref), [])
_v = dict(_bj[0]['verses'][3], text='पृथिव्यापस्तेजो वायुराकाशमेव च ।\nएतानि पञ्च भूतानि ॥')   # a verse the reference does not have
_p = ref_check([dict(verses=_bj[0]['verses'][:3] + [_v, dict(_v, n=5, text='क ···· ख')])], _ref); test_eq(len(_p), 3); assert 'lacuna' in _p[1]
_ref2 = {(ci, v['n']): letters(v['text']) for ci, c in enumerate(_bj, 1) for v in c['verses']}
_a, _b = _bj[0]['verses'][0], _bj[0]['verses'][1]
_moved = [dict(_a, text=_a['text'].split('\n')[0]), dict(_b, text=_a['text'].split('\n')[1] + '\n' + _b['text'])]   # verse 1's last line read as verse 2's first
_p = []; _m = move_lines([dict(verses=_moved)], _ref2, _p)[0]['verses']
test_eq([v['text'] for v in _m], [_a['text'], _b['text']]); test_eq(_p, ['1.2: 1 leading line(s) moved back to 1.1'])
_q = [_a, dict(_b, text='पृथिव्यापस्तेजो वायुराकाशमेव च ।\n' + _b['text'])]
_p = []; test_eq(move_lines([dict(verses=_q)], _ref2, _p)[0]['verses'][1]['text'], _b['text']); assert 'quotation' in _p[0]

## Two readings of a page

`verse_text` is a page's verses as bare letters (numbers, daṇḍas, spaces and avagraha gone); `numbered` keeps only the verses read with a number, leaving out a verse run over a page break or one a note quotes. `disagree` is how far two readings of a page are apart: character edits (`lev`) over the longer's letters, 1 when only one found verses, None when neither did or one is missing. `compare` tabulates it for each pair of named readings, a reading being a list of pages or `{page: page}`, as markdown with the pages the first pair flags at `at` (10%) or more; `pages` lists those pages alone.

In [ ]:
#| export
_STRICT = re.compile(r'[\s0-9०-९।॥|\-–.,:;ऽ\u200c\u200d()\[\]]')
def verse_text(page, numbered:bool=False) -> str:
    "A page's verses as bare letters; `numbered` keeps only those read with a number."
    if not page: return ''
    return _STRICT.sub('', ''.join(b.get('text') or '' for b in blocks(page) if kind(b) == 'verse' and not (numbered and b.get('n') is None)))

def lev(a:str, b:str) -> int:
    "Character edits (Levenshtein) between two strings."
    p = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        c = [i]
        for j, y in enumerate(b, 1): c.append(min(p[j] + 1, c[j - 1] + 1, p[j - 1] + (x != y)))
        p = c
    return p[-1]

def disagree(a, b, numbered:bool=False) -> float|None:
    "Two readings of a page apart, as edits over the longer's letters: 1 when only one found verses, None when neither did or one is missing."
    if a is None or b is None: return None
    x, y = verse_text(a, numbered), verse_text(b, numbered)
    return lev(x, y) / max(len(x), len(y)) if x or y else None

def compare(readings:dict, pairs, at:float=.1, pages=None, numbered:bool=False) -> tuple:
    "A markdown table of how far each pair of named readings is apart, page by page, and the pages the first pair flags at `at` or more."
    rd = {k: v if isinstance(v, dict) else {p['page']: p for p in v} for k, v in readings.items()}
    names = pages or sorted(set(rd[pairs[0][0]]) & set(rd[pairs[0][1]]))
    rows = [(n, [disagree(rd[a].get(n), rd[b].get(n), numbered) for a, b in pairs]) for n in names]
    flag = [n for n, rs in rows if rs[0] is not None and rs[0] >= at]
    f = lambda r: '—' if r is None else f'{r:.1%}'
    stat = lambda k: [r[k] for _, r in rows if r[k] is not None]
    md = ['| page | ' + ' | '.join(f'{a} vs {b}' for a, b in pairs) + ' |', '|---' * (len(pairs) + 1) + '|']
    md += [f'| {n} | ' + ' | '.join(map(f, rs)) + ' |' for n, rs in rows if pages or n in flag]
    md += ['', ', '.join(f'{a} vs {b}: {len(s)} pages, mean {sum(s) / max(1, len(s)):.1%}, {sum(x >= at for x in s)} at {at:.0%} or more' for k, (a, b) in enumerate(pairs) for s in [stat(k)])]
    return '\n'.join(md), flag

In [ ]:
test_eq(lev('kitten', 'sitting'), 3); test_eq(verse_text(dict(items=[dict(kind='verse', text='राम ॥ १ ॥'), dict(kind='prose', text='x'), dict(kind='verse', text='सीता')]), numbered=True), '')
_one = json.load(open('samples/brihat_jataka.json'))
_two = [dict(p, blocks=[dict(b, text=b['text'].replace('ग्र', 'ग')) for b in p['blocks']]) for p in _one]
_two[3] = dict(_two[3], blocks=[dict(b, text='अ') if b['kind'] == 'verse' else b for b in _two[3]['blocks']])
test_eq(disagree(_one[0], _one[0]), 0); assert 0 < disagree(_one[0], _two[0]) < .1; test_eq(disagree(_one[0], None), None)
_md, _fl = compare(dict(a=_one, b=_two), [('a', 'b')])
test_eq(_fl, [_one[3]['page']]); assert _md.startswith('| page | a vs b |') and '16 pages' in _md

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()